# Etapa 3 - Análise exploratória
**Projeto:** Banco do Brasil e ODS 8

O notebook organiza a leitura dos seis arquivos trimestrais do Banco Central, a verificação da qualidade dos dados, o recorte do Banco do Brasil e as visualizações usadas no storytelling.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'dados' / 'raw').exists():
    ROOT = ROOT.parent
RAW = ROOT / 'dados' / 'raw'
arquivos = sorted(RAW.glob('ranking_bcb_20*_T*.csv'))
print('Arquivos encontrados:', len(arquivos))
[p.name for p in arquivos]

## 1. Leitura e consolidação

In [ ]:
quadros = []
for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=';', encoding='utf-8-sig', dtype=str)
    df = df.loc[:, [c for c in df.columns if str(c).strip()]]
    df['arquivo_origem'] = arquivo.name
    quadros.append(df)
base = pd.concat(quadros, ignore_index=True, sort=False)
base.shape

## 2. Metadados e valores ausentes

In [ ]:
pd.DataFrame({
    'tipo': base.dtypes.astype(str),
    'nulos': base.isna().sum(),
    'percentual_nulos': (base.isna().mean()*100).round(2),
    'valores_distintos': base.nunique(dropna=True)
})

## 3. Conversão das variáveis numéricas

In [ ]:
def numero_br(valor):
    if pd.isna(valor): return np.nan
    texto = str(valor).strip()
    if not texto: return np.nan
    if ',' in texto and '.' in texto:
        texto = texto.replace('.', '').replace(',', '.')
    elif ',' in texto:
        texto = texto.replace(',', '.')
    texto = re.sub(r'[^0-9.\\-]', '', texto)
    try: return float(texto)
    except ValueError: return np.nan

numericas = [
 'Índice','Quantidade total de reclamações respondidas',
 'Quantidade de reclamações procedentes',
 'Quantidade de reclamações procedentes extrapoladas',
 'Quantidade total de reclamações analisadas',
 'Quantidade total de clientes – CCS e SCR'
]
for c in numericas:
    base[c] = base[c].map(numero_br)
base[numericas].describe().T

## 4. Banco do Brasil

In [ ]:
bb = base[base['Instituição financeira'].eq('BB (conglomerado)')].copy()
bb['Ano'] = pd.to_numeric(bb['Ano'])
bb['Trimestre_num'] = bb['Trimestre'].str.extract(r'([1-4])')[0].astype(int)
bb = bb.sort_values(['Ano','Trimestre_num'])
bb['Periodo'] = bb['Ano'].astype(str) + ' T' + bb['Trimestre_num'].astype(str)
bb[['Periodo','Índice','Quantidade de reclamações procedentes extrapoladas','Quantidade total de clientes – CCS e SCR']]

A coluna de clientes usada aqui é a base do Banco Central (CCS + SCR). Ela não deve ser confundida com o total corporativo de clientes divulgado pelo Banco do Brasil.

## 5. Estatística descritiva do BB

In [ ]:
bb[numericas].describe().T

## 6. Evolução do índice

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(bb['Periodo'], bb['Índice'], marker='o')
plt.title('Evolução do índice de reclamações do Banco do Brasil')
plt.xlabel('Período')
plt.ylabel('Índice de reclamações')
plt.grid(axis='y', alpha=.25)
plt.show()

## 7. Leitura dos resultados
No recorte analisado, o índice do BB variou de 16,22 a 32,68. O menor valor ocorreu em 2025 T4 e o maior em 2026 T2. A série oscila ao longo dos seis trimestres, portanto não é adequado descrevê-la como uma tendência linear contínua.